# 20260923 Port Signal Extraction

Use this notebook when a behavior recording has no Bpod byte file and port on/off timing has to be recovered from behavior-video pixel intensity.

This notebook is a checkpoint, not a replacement for dataframe construction. It lets you choose one recording, draw or load a port-light ROI, extract the ROI intensity trace, tune the event threshold, inspect the start/stop frames, and save a verified event table that dataframe construction can consume.

## Pipeline Notebook Map

- `20260919_neu_preprocess.ipynb`: check one miniscope AVI, convert AVI to local H5, run/track EXTRACT and ActSort handoff, import curated neurons.
- `20260919_dataframe_construction.ipynb`: build one aligned behavior/SLEAP/neural dataframe, collect arena ROIs, handle cue overrides, segment trials.
- `20260919_visualize.ipynb`: inspect one processed recording or compare processed recordings with trajectory, 2D ratemap, EBC, HD, and trial plots.
- `20260923_trial_classification.ipynb`: label trajectory trials, train sklearn classifiers, predict/review labels for unlabeled trials.
- `20260923_port_signal_extraction.ipynb`: fallback for recordings without Bpod bytes; recover port on/off intervals from a hand-drawn video ROI.

The notebooks stay interactive. The repeated calculations live in `preprocess_functions/` and the batchable steps live in `scripts/`.

## Imports

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import ipywidgets as widgets
from IPython.display import display, clear_output

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from preprocess_functions.manifest import load_session_records
from preprocess_functions.pipeline import behavior_output_dir, default_aligned_session_path
from preprocess_functions.roi import load_or_collect_roi, roi_json_path
from preprocess_functions.port_signal import (
    add_aligned_times_to_port_events,
    auto_signal_threshold,
    crop_frame_around_roi,
    detect_signal_intervals,
    extract_roi_intensity_signal,
    read_video_frame,
    smooth_port_signal,
    video_metadata,
    write_port_signal_outputs,
)

## Config

In [ ]:
OUTPUT_ROOT = Path("preprocess_out")
MANIFEST = OUTPUT_ROOT / "manifest_with_cells.csv"
if not MANIFEST.exists():
    MANIFEST = OUTPUT_ROOT / "manifest_with_h5.csv"

# Set this only when spreadsheet paths are lab-relative, for example "Z:/" or "/Volumes/ASA_Lab".
LAB_DRIVE = None

FPS = 30.0
PORT_ROI_FOLDER = "port_rois"
DEFAULT_PORTS = ["port_1", "port_2", "port_3", "port_4"]

print("manifest:", MANIFEST)
print("output root:", OUTPUT_ROOT.resolve())

## Load Recordings

In [ ]:
records = load_session_records(MANIFEST, lab_drive=LAB_DRIVE)
record_table = pd.DataFrame([
    {
        "recording_id": r.recording_id,
        "session_id": r.session_id,
        "mouse_id": r.mouse_id,
        "trial_type": r.trial_type,
        "beh_vid": str(r.beh_vid) if r.beh_vid else None,
        "has_bpod_ts": r.bpod_ts is not None,
        "video_port_events_csv": str(r.video_port_events_csv) if r.video_port_events_csv else None,
    }
    for r in records
])

display(record_table)
display(record_table.query("has_bpod_ts == False") if not record_table.empty else record_table)

## Choose One Recording

In [ ]:
record_options = record_table["recording_id"].dropna().tolist()
recording_dropdown = widgets.Dropdown(options=record_options, description="recording")
port_dropdown = widgets.Dropdown(options=DEFAULT_PORTS, description="port")
display(widgets.HBox([recording_dropdown, port_dropdown]))

def selected_record():
    return next(r for r in records if r.recording_id == recording_dropdown.value)

def show_record_summary(*_):
    record = selected_record()
    print("recording_id:", record.recording_id)
    print("behavior video:", record.beh_vid)
    print("bpod_ts:", record.bpod_ts)
    if record.beh_vid:
        try:
            print("video metadata:", video_metadata(record.beh_vid))
        except Exception as exc:
            print("video metadata error:", exc)

show_record_summary()

## Select Or Load Port ROI

Click **collect/load ROI**. If the ROI JSON already exists, it loads immediately. To redraw it, check **redraw** and run the button again. In the OpenCV window, left-click to add points, right-click to undo, press `s` to save, or `q` to quit.

In [ ]:
roi_state = {"points": None, "json_path": None}

redraw_checkbox = widgets.Checkbox(value=False, description="redraw")
collect_button = widgets.Button(description="collect/load ROI", button_style="primary")
roi_output = widgets.Output()

def plot_roi_preview(record, port_name, points):
    frame = read_video_frame(record.beh_vid, frame_idx=0)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.imshow(frame)
    ax.add_patch(Polygon(np.asarray(points), closed=True, fill=False, edgecolor="lime", linewidth=2))
    ax.set_title(f"{record.recording_id} {port_name} ROI")
    ax.axis("off")
    plt.show()

def on_collect_roi(_):
    with roi_output:
        clear_output(wait=True)
        record = selected_record()
        port_name = port_dropdown.value
        path = roi_json_path(record.recording_id, port_name, root=PROJECT_ROOT, folder_name=PORT_ROI_FOLDER)
        if redraw_checkbox.value and path.exists():
            path.unlink()
            print("deleted old ROI:", path)
        points, json_path = load_or_collect_roi(
            video_path=record.beh_vid,
            roi_name=port_name,
            session_id=record.recording_id,
            root=PROJECT_ROOT,
            folder_name=PORT_ROI_FOLDER,
        )
        roi_state["points"] = np.asarray(points, dtype=float)
        roi_state["json_path"] = json_path
        print("ROI:", json_path)
        plot_roi_preview(record, port_name, roi_state["points"])

collect_button.on_click(on_collect_roi)
display(widgets.HBox([collect_button, redraw_checkbox]), roi_output)

## Extract Signal And Detect Events

In [ ]:
signal_state = {"signal": None, "events": None, "threshold": None}

frame_step_widget = widgets.IntText(value=1, description="frame step")
max_frames_widget = widgets.IntText(value=0, description="max frames")
smooth_widget = widgets.IntSlider(value=5, min=1, max=101, step=2, description="smooth")
min_on_widget = widgets.IntSlider(value=2, min=1, max=60, step=1, description="min on")
threshold_widget = widgets.FloatText(value=np.nan, description="threshold")
extract_button = widgets.Button(description="extract signal", button_style="primary")
detect_button = widgets.Button(description="detect events")
signal_output = widgets.Output()

def run_detection():
    signal = signal_state["signal"]
    signal = smooth_port_signal(signal, smooth_window=smooth_widget.value)
    threshold = threshold_widget.value
    if not np.isfinite(threshold):
        threshold = auto_signal_threshold(signal)
        threshold_widget.value = threshold
    events = detect_signal_intervals(
        signal,
        threshold=threshold,
        port_name=port_dropdown.value,
        min_on_frames=min_on_widget.value,
    )
    record = selected_record()
    aligned_path = default_aligned_session_path(OUTPUT_ROOT, record)
    if aligned_path.exists() and not events.empty:
        aligned = pd.read_csv(aligned_path)
        events = add_aligned_times_to_port_events(events, aligned)
    signal_state.update({"signal": signal, "events": events, "threshold": threshold})
    return signal, events, threshold

def plot_signal(signal, events, threshold):
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(signal["frame_idx"], signal["roi_mean"], color="0.7", linewidth=0.8, label="raw")
    ax.plot(signal["frame_idx"], signal["roi_mean_smooth"], color="black", linewidth=1.2, label="smooth")
    ax.axhline(threshold, color="crimson", linestyle="--", linewidth=1.0, label="threshold")
    for _, event in events.iterrows():
        ax.axvspan(event["start_frame"], event["stop_frame"], color="gold", alpha=0.25)
    ax.set_xlabel("behavior video frame")
    ax.set_ylabel("ROI intensity")
    ax.legend(loc="upper right")
    plt.show()

def on_extract(_):
    with signal_output:
        clear_output(wait=True)
        if roi_state["points"] is None:
            raise RuntimeError("Collect or load a port ROI first.")
        record = selected_record()
        max_frames = max_frames_widget.value if max_frames_widget.value > 0 else None
        signal = extract_roi_intensity_signal(
            record.beh_vid,
            roi_state["points"],
            frame_step=frame_step_widget.value,
            max_frames=max_frames,
        )
        signal_state["signal"] = signal
        threshold_widget.value = np.nan
        signal, events, threshold = run_detection()
        print(f"signal rows: {len(signal)}")
        print(f"detected events: {len(events)}")
        display(events.head(20))
        plot_signal(signal, events, threshold)

def on_detect(_):
    with signal_output:
        clear_output(wait=True)
        if signal_state["signal"] is None:
            raise RuntimeError("Extract a signal first.")
        signal, events, threshold = run_detection()
        print(f"detected events: {len(events)}")
        display(events.head(20))
        plot_signal(signal, events, threshold)

extract_button.on_click(on_extract)
detect_button.on_click(on_detect)
display(widgets.VBox([
    widgets.HBox([frame_step_widget, max_frames_widget]),
    widgets.HBox([smooth_widget, min_on_widget, threshold_widget]),
    widgets.HBox([extract_button, detect_button]),
]), signal_output)

## Review Start/Stop Frames

Use this GUI to check whether each detected event begins and ends on the correct frames. Adjust the threshold above, re-detect, and revisit this section until the events look right.

In [ ]:
event_slider = widgets.IntSlider(value=0, min=0, max=0, step=1, description="event")
start_frame_widget = widgets.IntText(value=0, description="start")
stop_frame_widget = widgets.IntText(value=0, description="stop")
review_button = widgets.Button(description="refresh review")
apply_edit_button = widgets.Button(description="apply frame edit", button_style="warning")
review_output = widgets.Output()

def plot_event_review(event_number=0):
    with review_output:
        clear_output(wait=True)
        signal = signal_state["signal"]
        events = signal_state["events"]
        threshold = signal_state["threshold"]
        if signal is None or events is None or events.empty:
            print("No detected events yet.")
            return

        event_number = int(np.clip(event_number, 0, len(events) - 1))
        event = events.iloc[event_number]
        event_slider.max = max(0, len(events) - 1)
        event_slider.value = event_number
        record = selected_record()

        start_frame = int(event["start_frame"])
        stop_frame = int(event["stop_frame"])
        start_frame_widget.value = start_frame
        stop_frame_widget.value = stop_frame
        start_img = read_video_frame(record.beh_vid, start_frame)
        stop_img = read_video_frame(record.beh_vid, stop_frame)
        start_crop, start_roi = crop_frame_around_roi(start_img, roi_state["points"], padding=30)
        stop_crop, stop_roi = crop_frame_around_roi(stop_img, roi_state["points"], padding=30)

        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        ax = axes[0]
        ax.plot(signal["frame_idx"], signal["roi_mean_smooth"], color="black", linewidth=1)
        ax.axhline(threshold, color="crimson", linestyle="--", linewidth=1)
        ax.axvline(start_frame, color="green", linewidth=1.5, label="start")
        ax.axvline(stop_frame, color="red", linewidth=1.5, label="stop")
        ax.set_title(f"event {event_number}: frames {start_frame}-{stop_frame}")
        ax.set_xlabel("frame")
        ax.set_ylabel("smoothed ROI intensity")
        ax.legend(loc="upper right")

        axes[1].imshow(start_crop)
        axes[1].add_patch(Polygon(start_roi, closed=True, fill=False, edgecolor="lime", linewidth=2))
        axes[1].set_title(f"start frame {start_frame}")
        axes[1].axis("off")

        axes[2].imshow(stop_crop)
        axes[2].add_patch(Polygon(stop_roi, closed=True, fill=False, edgecolor="lime", linewidth=2))
        axes[2].set_title(f"stop frame {stop_frame}")
        axes[2].axis("off")
        plt.show()

        display(pd.DataFrame([event]))

def on_event_change(change):
    if change["name"] == "value":
        plot_event_review(change["new"])

def on_refresh_review(_):
    plot_event_review(event_slider.value)

def on_apply_frame_edit(_):
    events = signal_state["events"]
    signal = signal_state["signal"]
    if events is None or signal is None or events.empty:
        return
    event_number = int(event_slider.value)
    start_frame = int(start_frame_widget.value)
    stop_frame = int(stop_frame_widget.value)
    if stop_frame < start_frame:
        with review_output:
            print("stop frame must be >= start frame")
        return
    edited = events.copy()
    row_index = edited.index[event_number]
    edited.loc[row_index, "start_frame"] = start_frame
    edited.loc[row_index, "stop_frame"] = stop_frame
    edited.loc[row_index, "last_active_frame"] = max(start_frame, stop_frame - 1)
    edited.loc[row_index, "duration_frames"] = max(0, stop_frame - start_frame)
    fps_values = pd.to_numeric(signal.get("video_fps"), errors="coerce").dropna()
    if not fps_values.empty and fps_values.iloc[0] > 0:
        edited.loc[row_index, "duration_s"] = edited.loc[row_index, "duration_frames"] / fps_values.iloc[0]
    record = selected_record()
    aligned_path = default_aligned_session_path(OUTPUT_ROOT, record)
    if aligned_path.exists():
        edited = add_aligned_times_to_port_events(edited, pd.read_csv(aligned_path))
    signal_state["events"] = edited
    plot_event_review(event_number)

event_slider.observe(on_event_change)
review_button.on_click(on_refresh_review)
apply_edit_button.on_click(on_apply_frame_edit)
display(widgets.VBox([
    widgets.HBox([event_slider, review_button]),
    widgets.HBox([start_frame_widget, stop_frame_widget, apply_edit_button]),
]), review_output)

## Save Verified Events

This writes per-port files plus one combined `video_port_events.csv` for the recording. Re-run dataframe construction afterward so the aligned session CSV gets `video_port_*_active` columns.

In [ ]:
save_button = widgets.Button(description="save verified events", button_style="success")
save_output = widgets.Output()

def on_save(_):
    with save_output:
        clear_output(wait=True)
        if signal_state["signal"] is None or signal_state["events"] is None:
            raise RuntimeError("Extract and review events before saving.")
        record = selected_record()
        paths = write_port_signal_outputs(
            behavior_output_dir(OUTPUT_ROOT, record),
            record.recording_id,
            port_dropdown.value,
            signal_state["signal"],
            signal_state["events"],
        )
        for key, path in paths.items():
            print(f"{key}: {path}")
        print("\nNext: rerun scripts/build_aligned_sessions.py for this recording to add video port columns.")

save_button.on_click(on_save)
display(save_button, save_output)

## Compare Saved Video-Derived Port Events

In [ ]:
event_files = sorted(OUTPUT_ROOT.glob("*/behavior/*_video_port_events.csv"))
all_video_port_events = []
for path in event_files:
    df = pd.read_csv(path)
    df["events_csv"] = str(path)
    all_video_port_events.append(df)

if all_video_port_events:
    all_video_port_events = pd.concat(all_video_port_events, ignore_index=True)
    summary = (
        all_video_port_events
        .groupby(["recording_id", "port_name"], dropna=False)
        .agg(n_events=("start_frame", "size"), median_duration_s=("duration_s", "median"))
        .reset_index()
    )
    display(summary)
    ax = summary.pivot(index="recording_id", columns="port_name", values="n_events").fillna(0).plot(kind="bar", figsize=(12, 4))
    ax.set_ylabel("detected events")
    plt.tight_layout()
    plt.show()
else:
    print("No saved video-derived port event files found yet.")